## 선택 · 심화 문제 1. Offset mapping으로 원문 범위 추적

### 문제 배경

오류 분석에서는 어떤 subword가 원문의 어느 문자 범위에서 왔는지 알아야 할 수 있습니다. Fast tokenizer의 `offset_mapping`을 token·ID와 나란히 정리합니다.

### 시작 코드

```python
text = "한국어 토큰화 테스트"

def build_offset_table(text):
    raise NotImplementedError
```

### 수행 요구사항

1. Fast tokenizer인지 확인하고 아니면 명시적 오류를 내세요.
2. `return_offsets_mapping=True`로 한 문장을 encode하세요.
3. 각 위치에 `position/token/id/start/end/source_piece/is_special`을 기록하세요.
4. Non-special token의 `source_piece == text[start:end]`를 검증하세요.

### 제출 결과

- 위치별 offset table
- Special token의 offset 처리 설명
- Normalization 때문에 offset을 해석할 때 주의할 점
- `심화 문제 1 자동 검증: PASS`

### 자동 검증

```python
table = build_offset_table(text)
content = [row for row in table if not row["is_special"]]
assert content and all(row["source_piece"] == text[row["start"]:row["end"]] for row in content)
assert table[0]["is_special"] and table[-1]["is_special"]
print("심화 문제 1 자동 검증: PASS")
```

    ```
    
   **상세 해설** · Special token은 원문에서 온 문자가 아니므로 보통 `(0,0)` offset입니다. Fast tokenizer의 offset은 원문 추적에 유용하지만 Unicode normalization과 공백 처리 규칙을 확인해야 합니다.
    
   **자주 하는 실수**
    
    - Special token의 `(0,0)`을 첫 글자 범위로 해석합니다.
    - Slow tokenizer에서도 offset이 항상 지원된다고 가정합니다.
    - `##` 표기를 원문 substring 일부로 간주합니다.

In [7]:
from transformers import AutoTokenizer
MODEL_ID = "monologg/koelectra-small-v3-discriminator"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

config.json:   0%|          | 0.00/458 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/61.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/263k [00:00<?, ?B/s]

In [8]:
from transformers import AutoTokenizer

MODEL_ID = "monologg/koelectra-small-v3-discriminator"
text = "한국어 토큰화 테스트"

def build_offset_table(text):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, local_files_only=True)
    # Offset mapping은 Rust 기반 fast tokenizer에서만 제공되는 기능입니다.
    if not tokenizer.is_fast:
        raise RuntimeError("offset_mapping에는 fast tokenizer가 필요합니다.")
    encoded = tokenizer(text, add_special_tokens=True, return_offsets_mapping=True)
    tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])
    special_ids = set(tokenizer.all_special_ids)
    # Token, ID, 원문 범위를 같은 index로 묶어 사람이 검사할 표를 만듭니다.
    table = []
    for position, (token, token_id, offset) in enumerate(
        zip(tokens, encoded["input_ids"], encoded["offset_mapping"])
    ):
        start, end = offset
        is_special = token_id in special_ids
        table.append({
            "position": position, "token": token, "id": token_id,
            "start": start, "end": end,
            "source_piece": "" if is_special else text[start:end],
            "is_special": is_special,
        })
    return table

table = build_offset_table(text)
for row in table: print(row)
content = [row for row in table if not row["is_special"]]
assert content and all(row["source_piece"] == text[row["start"]:row["end"]] for row in content)
assert table[0]["is_special"] and table[-1]["is_special"]
print("심화 문제 1 자동 검증: PASS")

{'position': 0, 'token': '[CLS]', 'id': 2, 'start': 0, 'end': 0, 'source_piece': '', 'is_special': True}
{'position': 1, 'token': '한국어', 'id': 11229, 'start': 0, 'end': 3, 'source_piece': '한국어', 'is_special': False}
{'position': 2, 'token': '토큰', 'id': 32436, 'start': 4, 'end': 6, 'source_piece': '토큰', 'is_special': False}
{'position': 3, 'token': '##화', 'id': 4162, 'start': 6, 'end': 7, 'source_piece': '화', 'is_special': False}
{'position': 4, 'token': '테스트', 'id': 9380, 'start': 8, 'end': 11, 'source_piece': '테스트', 'is_special': False}
{'position': 5, 'token': '[SEP]', 'id': 3, 'start': 0, 'end': 0, 'source_piece': '', 'is_special': True}
심화 문제 1 자동 검증: PASS
